# Setup FDSN client

In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

t0 = UTCDateTime("2009-03-20T00:00:00")
t1 = UTCDateTime("2009-03-23T00:00:00")


# Get events

In [ ]:

# a simple geographic box around Redoubt (tweak)
usgs = Client("USGS")  # queries earthquake.usgs.gov FDSN event service  [oai_citation:3‡USGS](https://earthquake.usgs.gov/fdsnws/event/1/?utm_source=chatgpt.com)

cat = usgs.get_events(
    starttime=t0, endtime=t1,
    minlatitude=60.2, maxlatitude=60.8,
    minlongitude=-153.2, maxlongitude=-152.2,
    # optionally: minmagnitude=1.5,
)

cat.write("comcat_redoubt_20090320_20090322.quakeml", format="QUAKEML")
print(cat)

# Get station metadata

In [ ]:
client = Client("IRIS")  # or "IRIS"

inv = client.get_stations(
    network="AV",
    starttime=UTCDateTime("2009-03-20"),
    endtime=UTCDateTime("2009-03-23"),
    level="response",  # full StationXML response stage info
)
inv.write("AV_20090320_20090322_station.xml", format="STATIONXML")
print(inv)

# Get waveform data

In [ ]:
# Optional: restrict to common seismic channels
keep_prefixes = ("BH", "EH", "HH", "SH")   # broadband / short-period families

# 2) Build a list of channel codes to request (net, sta, loc, cha)
chans = []
for net in inv:
    for sta in net:
        for cha in sta:
            if not cha.code.startswith(keep_prefixes):
                continue
            loc = cha.location_code or ""
            chans.append((net.code, sta.code, loc, cha.code))

# De-duplicate (inventory sometimes repeats channels across epochs)
chans = sorted(set(chans))

# 3) Download and write daily files
out_dir = "redoubt_20090320_20090322_daily"
import os
print(os.getcwd())
os.makedirs(out_dir, exist_ok=True)

day = t0
while day < t1:
    day_end = day + 24 * 3600

    # Download one day for all discovered channels (bulk request is faster)
    st = client.get_waveforms_bulk(
        bulk=[(n, s, l, c, day, day_end) for (n, s, l, c) in chans],
        attach_response=False,
    )

    # Merge/fill small overlaps; keep gaps as gaps
    try:
        st.merge(method=1, fill_value=None)
    except Exception as e:
        print(f"Warning: merge failed for day {day.date} with error: {e}")

    # Write one 24h file per station per day (contains all channels for that station)
    for (net, sta) in sorted({(tr.stats.network, tr.stats.station) for tr in st}):
        st_sta = st.select(network=net, station=sta)
        if len(st_sta) == 0:
            continue

        fname = f"{net}.{sta}.{day.date}.mseed"   # e.g., AV.RDT.2009-03-20.mseed
        full_path = os.path.join(out_dir, fname)
        print(f"Writing {full_path} with {len(st_sta)} traces...")
        st_sta.write(full_path, format="MSEED")

        if sta == "REF":
            for tr in st_sta:
                tr.plot(type="dayplot");

    day = day_end

print(f"Done. Wrote daily files into: {out_dir}")